# 07 · Profile: industry jobs, pay and output in NYC (QCEW, CES, BEA)

**Method:** `docs/framework.md` §3 (the 10-section exploration template). Cards: `docs/source-cards/nysdol_qcew.md`, `nysdol_ces.md`, `bea_county_gdp.md`.

**Why these sources matter.** brief S4 gives users two ways into a job: by major and by industry. The industry side needs to say which NYC industries are big, growing and paying, and where (borough). Three government sources measure that, each in a different way:

| | QCEW | CES | BEA county GDP |
|---|---|---|---|
| Measures | Jobs, employers, wages | Jobs | Output (GDP) |
| How | **Count** of every unemployment-insurance account | Monthly survey of employers, benchmarked to QCEW | **Model** built from QCEW wages + other inputs |
| Area | Each borough + NYC | NYC | Each borough |
| Time | Year (2000–2025) | Month (1990–now) | Year (2001–2024) |

**What this notebook does:** reads each file, checks grain, codes and suppression, ties the three to each other and to the publishers, and ends with an issues list and proposed staging tables. It changes nothing. The industry → job bridge (BLS matrix) is notebook 09; hiring by industry (QWI) is notebook 08.

Every read uses `dtype=str, keep_default_na=False`.

In [1]:
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)

BOROUGHS = {"36005": "Bronx", "36047": "Brooklyn (Kings)", "36061": "Manhattan (New York)",
            "36081": "Queens", "36085": "Staten Island (Richmond)"}
BORO_COUNTY = ["Bronx County", "Kings County", "New York County", "Queens County", "Richmond County"]
YEARS = [str(y) for y in range(2019, 2026)]  # brief scope: 2019 -> latest

man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
display(man[man.key.isin(["nysdol_qcew_annual", "nysdol_ces", "bea_county_gdp"])][["key", "file", "downloaded_utc", "bytes", "note"]])

,key,file,downloaded_utc,bytes,note
13,nysdol_qcew_annual,jobs/nysdol_qcew_annual/shc7-xcbw.csv,2026-10-12T17:04:29+00:00,128007867,"QCEW annual jobs/wages by county x NAICS, 2000-2025"
14,nysdol_ces,jobs/nysdol_ces/6k74-dgkb.csv,2026-10-12T17:05:24+00:00,37064853,CES monthly jobs by industry since 1990 (NYC area 93561)
146,bea_county_gdp,jobs/bea_county_gdp/CAGDP2_2026-10-12.zip,2026-10-12T17:26:24+00:00,15288721,"GDP by county x industry, current dollars (NYC = 5 borough FIPS); ..."
147,bea_county_gdp,jobs/bea_county_gdp/CAGDP9_2026-10-12.zip,2026-10-12T17:26:25+00:00,15422725,"Real GDP by county x industry, chained dollars; BEA revises every ..."


---
# Part A · QCEW (Quarterly Census of Employment and Wages), annual

## A0. Source header

| | **NYS DOL QCEW annual averages** (`nysdol_qcew_annual`) |
|---|---|
| **Publisher** | NYS Department of Labor, Division of Research & Statistics, with BLS (QCEW is a federal–state program) |
| **Dataset** | *Quarterly Census of Employment and Wages: Annual Data* (data.ny.gov [`shc7-xcbw`](https://data.ny.gov/d/shc7-xcbw)) |
| **How it was collected** | **Administrative records**, not a survey: every employer covered by unemployment insurance files quarterly counts and wages. It covers about 97% of wage and salary jobs ([BLS QCEW overview](https://www.bls.gov/cew/overview.htm)) |
| **Who's missing** | Self-employed, unpaid family workers, most farm workers, members of the armed forces, some religious-organization and railroad workers |
| **Codes** | NAICS (the edition current in each year: 2017 for 2017–21, 2022 from 2022) |
| **Confidentiality** | Cells that would identify an employer are not published. In this extract they're left out as rows, not shown as a symbol (checked below) |
| **License** | NY Open Data |

In [2]:
QCEW = RAW / man.loc[man.key == "nysdol_qcew_annual", "file"].iloc[0]
q = pd.read_csv(QCEW, dtype=str, keep_default_na=False)
print(f"{len(q):,} rows x {q.shape[1]} cols")
print("Column names with stray spaces:", [c for c in q.columns if c != c.strip()])
q.columns = [c.strip() for c in q.columns]
NUM_Q = ["Establishments", "Average Employment", "Total Wage", "Annual Average Salary"]
print("Non-digit cells in the numeric columns:", {c: int((~q[c].str.fullmatch(r"\d+")).sum()) for c in NUM_Q})
print("Text cells with stray spaces:", {c: int((q[c] != q[c].str.strip()).sum()) for c in q.columns if (q[c] != q[c].str.strip()).any()} or "none")
display(q.head(3))

1,163,386 rows x 10 cols
Column names with stray spaces: ['Annual Average Salary                                                                                                                                                  ']


Non-digit cells in the numeric columns: {'Establishments': 0, 'Average Employment': 0, 'Total Wage': 0, 'Annual Average Salary': 0}


Text cells with stray spaces: none


,Area Type,Area,Ownership,NAICS,NAICS Title,Year,Establishments,Average Employment,Total Wage,Annual Average Salary
0,State,New York State,Private,11,"Agriculture, Forestry, Fishing and Hunting",2025,3401,29576,1458136146,49301
1,State,New York State,"Total, All Ownerships",11,"Agriculture, Forestry, Fishing and Hunting",2025,3401,29576,1458136146,49301
2,State,New York State,Private,111,Crop Production,2025,1508,13622,594867687,43670


**Load findings**
- 1,163,386 rows × 10 columns, all clean digits in the four numeric columns: no suppression symbols at all.
- One column name, `Annual Average Salary`, carries ~150 trailing spaces. Cells have no stray spaces.

## A2. What one row is (grain)

**Presumed key:** `Area + Ownership + NAICS + Year`.

In [3]:
k4 = ["Area", "Ownership", "NAICS", "Year"]
k5 = ["Area Type"] + k4
print("Duplicates on area + ownership + NAICS + year:", q.duplicated(k4).sum())
print("Duplicates once Area Type is added:            ", q.duplicated(k5).sum())
multi = q[["Area Type", "Area"]].drop_duplicates().groupby("Area")["Area Type"].agg(list)
print("\nArea names listed under more than one area type:"); display(multi[multi.str.len() > 1])

# Are the New York City copies identical?
nycs = q[q.Area == "New York City"]
wide = nycs.pivot_table(index=k4[1:], columns="Area Type", values="Average Employment", aggfunc="first")
print("NYC rows per area type:", nycs["Area Type"].value_counts().to_dict())
print("NYC copies identical in every row they share:", (wide.nunique(axis=1) == 1).all())

Duplicates on area + ownership + NAICS + year: 21752


Duplicates once Area Type is added:             0

Area names listed under more than one area type:


Area
New York City    [Labor Market Area, Workforce Investment Region, Metropolitan Stat...
Name: Area Type, dtype: object

NYC rows per area type: {'Labor Market Area': 18885, 'Metropolitan Statistical Area': 18885, 'Workforce Investment Region': 3194}


NYC copies identical in every row they share: True


**Grain findings**
- The real key is `Area Type + Area + Ownership + NAICS + Year` (unique ). Without `Area Type` there are 21,752 duplicates.
- They're all New York City, which the file lists three times: as a *Labor Market Area*, a *Metropolitan Statistical Area* and a *Workforce Investment Region*. The copies are identical wherever they overlap (the WIR copy has fewer rows). Staging keeps one: `Area Type = 'Labor Market Area'`.

## A3. Column profile

In [4]:
print("Area types:", q["Area Type"].value_counts().to_dict())
print("Ownership:", q.Ownership.value_counts().to_dict())
print("NAICS code length:", q.NAICS.str.len().value_counts().sort_index().to_dict())
print("Rows per year:"); print(q.Year.value_counts().sort_index().to_dict())
for c in NUM_Q:
    v = pd.to_numeric(q[c])
    print(f"{c:<22} min {v.min():>16,} | median {v.median():>14,.0f} | max {v.max():>18,} | zeros {(v == 0).sum():,}")

Area types: {'County': 666122, 'Metropolitan Statistical Area': 195640, 'Labor Market Area': 182165, 'Workforce Investment Region': 62613, 'Metropolitan Division': 34475, 'State': 22371}
Ownership: {'Private': 554952, 'Total, All Ownerships': 499123, 'Local Government': 36441, 'Total, All Government': 27578, 'Federal Government': 24083, 'State Government': 21209}


NAICS code length: {2: 116886, 3: 323965, 4: 722535}
Rows per year:


{'2000': 43473, '2001': 43309, '2002': 43080, '2003': 43094, '2004': 43162, '2005': 43027, '2006': 43078, '2007': 42665, '2008': 42723, '2009': 42495, '2010': 42477, '2011': 42534, '2012': 42682, '2013': 42570, '2014': 42768, '2015': 43082, '2016': 42966, '2017': 43130, '2018': 43323, '2019': 43004, '2020': 42546, '2021': 42662, '2022': 54567, '2023': 54510, '2024': 55310, '2025': 55149}


Establishments         min                1 | median             24 | max            711,853 | zeros 0


Average Employment     min                1 | median            325 | max          9,791,226 | zeros 0
Total Wage             min                0 | median     13,148,866 | max    978,627,408,241 | zeros 6


Annual Average Salary  min                0 | median         40,732 | max          3,759,114 | zeros 6


**Profile findings**
- Six area types, six ownership values (three of them are totals: *Total, All Ownerships*, *Total, All Government*, and the NAICS `00` line).
- NAICS at 2, 3 and 4 digits; no 5–6 digit detail. `00` = all industries, `99` = unclassified.
- About 43,000 rows a year until 2021, then ~55,000 from 2022: more area types or detail were added in 2022 (NYC itself got *fewer* detailed rows, see below).
- Six rows have `Total Wage = 0` with employment > 0: tiny cells, harmless.

## A4. Codes, hierarchy and special values

In [5]:
two = q[q.NAICS.str.len() == 2][["NAICS", "NAICS Title"]].drop_duplicates().sort_values("NAICS")
print("2-digit codes and their titles (a code with more than one title is a NAICS edition change):")
display(two.groupby("NAICS")["NAICS Title"].agg(lambda s: " | ".join(sorted(set(s)))).to_frame())

# Does each 3-digit code roll up into a 2-digit sector present in the file? NAICS ranges 31-33, 44-45, 48-49
SECTOR = {"32": "31", "33": "31", "45": "44", "49": "48"}
q["sector"] = q.NAICS.str[:2].replace(SECTOR)
print("Sectors referenced by 3/4-digit codes but absent as a 2-digit row:",
      sorted(set(q.loc[q.NAICS.str.len() > 2, "sector"]) - set(q.loc[q.NAICS.str.len() == 2, "NAICS"])))

# Edition check: 4-digit codes that appear or vanish at 2022 in NYC (NAICS 2017 -> 2022)
nyc = q[(q.Area == "New York City") & (q["Area Type"] == "Labor Market Area")].copy()
c21 = set(nyc[(nyc.Year == "2021")].NAICS); c22 = set(nyc[(nyc.Year == "2022")].NAICS)
print(f"\nNYC codes only in 2021: {len(c21 - c22)} {sorted(c21 - c22)[:20]}")
print(f"NYC codes only in 2022: {len(c22 - c21)} {sorted(c22 - c21)[:20]}")
print("NYC rows per year by code length:")
display(nyc.groupby(["Year", nyc.NAICS.str.len()]).size().unstack().loc[["2019", "2020", "2021", "2022", "2023", "2024", "2025"]])

2-digit codes and their titles (a code with more than one title is a NAICS edition change):


,NAICS Title
NAICS,
00,"Total, All Industries"
11,"Agriculture, Forestry, Fishing and Hunting"
21,Mining
22,Utilities
23,Construction
31,Manufacturing
42,Wholesale Trade
44,Retail Trade
48,Transportation and Warehousing


Sectors referenced by 3/4-digit codes but absent as a 2-digit row: []

NYC codes only in 2021: 39 ['3259', '442', '4421', '4422', '443', '4431', '446', '4461', '447', '4471', '448', '4481', '4482', '4483', '451', '4511', '4512', '452', '4522', '4523']
NYC codes only in 2022: 9 ['1113', '1153', '3211', '3336', '516', '5161', '5171', '927', '9271']
NYC rows per year by code length:


NAICS,2,3,4
Year,,,
2019,46,179,519
2020,44,182,525
2021,44,180,522
2022,50,171,491
2023,50,169,495
2024,51,171,494
2025,53,173,499


**Suppression.** The file has no suppression symbol, so a suppressed cell is a missing row. Test: within NYC, does the sum of the 2-digit sectors equal the `00` total? A gap is the employment hidden by suppression (plus `99` unclassified, which is in the file).

In [6]:
def gap(df, own):
    d = df[(df.Ownership == own)].copy()
    d["emp"] = pd.to_numeric(d["Average Employment"])
    tot = d[d.NAICS == "00"].groupby("Year").emp.sum()
    sec = d[(d.NAICS.str.len() == 2) & (d.NAICS != "00")].groupby("Year").emp.sum()
    return pd.DataFrame({"total": tot, "sum_of_sectors": sec, "hidden": tot - sec, "hidden_pct": (tot - sec) / tot * 100}).loc[YEARS]

print("NYC, Private:"); display(gap(nyc, "Private").round(2))
print("NYC, Total all ownerships:"); display(gap(nyc, "Total, All Ownerships").round(2))
# Ownership arithmetic: private + all government = total
o = nyc[nyc.NAICS == "00"].pivot_table(index="Year", columns="Ownership", values="Average Employment", aggfunc="first").astype(int)
o["private_plus_gov_minus_total"] = o["Private"] + o["Total, All Government"] - o["Total, All Ownerships"]
o["fed_state_local_minus_gov"] = o["Federal Government"] + o["State Government"] + o["Local Government"] - o["Total, All Government"]
print("Ownership arithmetic at the all-industry level (0 = adds up):")
display(o.loc[o.index >= "2015", ["private_plus_gov_minus_total", "fed_state_local_minus_gov"]].T)

# Which ownership rows exist at sector level? (NYC, 2024)
pres = nyc[(nyc.Year == "2024") & (nyc.NAICS.str.len() == 2)].pivot_table(
    index="NAICS", columns="Ownership", values="Average Employment", aggfunc="first")
print("NYC 2024, sector x ownership (blank = no row in the file):")
display(pres.fillna(""))

NYC, Private:


,total,sum_of_sectors,hidden,hidden_pct
Year,,,,
2019,3916345,3916346,-1,-0.0
2020,3421539,3421540,-1,-0.0
2021,3506836,3506836,0,0.0
2022,3793900,3793902,-2,-0.0
2023,3928359,3928360,-1,-0.0
2024,4063618,4063615,3,0.0
2025,4074085,4074087,-2,-0.0


NYC, Total all ownerships:


,total,sum_of_sectors,hidden,hidden_pct
Year,,,,
2019,4492708,900010,3592698,79.97
2020,3997689,730552,3267137,81.73
2021,4065781,746007,3319774,81.65
2022,4347360,806725,3540635,81.44
2023,4485695,821594,3664101,81.68
2024,4630197,822042,3808155,82.25
2025,4652929,825403,3827526,82.26


Ownership arithmetic at the all-industry level (0 = adds up):


Year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Ownership,,,,,,,,,,,
private_plus_gov_minus_total,0,0,0,0,0,0,0,0,-1,0,-1
fed_state_local_minus_gov,0,0,0,0,0,0,0,-1,1,0,0


NYC 2024, sector x ownership (blank = no row in the file):


Ownership,Federal Government,Local Government,Private,State Government,"Total, All Government","Total, All Ownerships"
NAICS,,,,,,
00,45969,485814,4063618,34796,566579,4630197
11,,,483,,,483
21,,,115,,,115
22,,,15975,,,
23,,,139379,,,
31,,,54366,,,54366
42,,,122357,,,122357
44,61,,294934,,61,294995
48,16319,55174,173358,,,


Why the "Total, All Ownerships" sector sum is only 18% of the total. Below the all-industry line, the file does not publish a "Total, All Ownerships" row for most sectors (construction, finance, health care…), even where only private employers exist. Sector totals have to be built from Private + Federal + State + Local, not read from the Total rows. The Private sector rows are complete: they add to the private total within ±3 jobs (rounding) every year.

In [7]:
# NAICS 2022 check: NAICS 2022 replaced retail 3-digit codes (442, 443, 446, 447, 448, 451, 452, 453, 454 -> 449, 455-459)
# and information (511, 515, 519 -> 513, 516, 517, 518, 519). Which ones does the file carry in 2024, all of NY State?
st24 = q[(q.Area == "New York State") & (q.Year == "2024") & (q.Ownership == "Private")]
print("NY State 2024 private, retail and information 3-digit codes present:")
display(st24[st24.NAICS.str.fullmatch(r"4[45]\d|51\d")][["NAICS", "NAICS Title", "Average Employment"]])
ret = st24.set_index("NAICS")["Average Employment"].astype(int)
print(f"Retail 3-digit codes cover {ret[['441', '444', '445']].sum() / ret['44']:.1%} of NY State retail jobs in 2024")

NY State 2024 private, retail and information 3-digit codes present:


,NAICS,NAICS Title,Average Employment
55509,441,Motor Vehicle and Parts Dealers,78299
55517,444,Building Material & Garden Supply Stores,65227
55523,445,Food and Beverage Stores,212732
55621,512,Motion Picture & Sound Recording Ind,52089
55626,516,Internet Publishing and Broadcasting,51147
55630,517,Telecommunications,33246
55636,518,"ISPs, Search Portals, & Data Processing",29039
55639,519,Other Information Services,30798


Retail 3-digit codes cover 42.7% of NY State retail jobs in 2024


**Codes findings**
- 2-digit sectors use the *first* code of NAICS ranges: `31` = 31–33 Manufacturing, `44` = 44–45 Retail, `48` = 48–49 Transportation. Every 3/4-digit code rolls up to a 2-digit row that exists.
- Below the sector level, the file doesn't follow NAICS 2022. NAICS 2022 rebuilt retail (442–454 → 449, 455–459) and information (511, 515 → 513, 516). In 2022+ the file simply drops the new retail codes: only 441, 444 and 445 remain, covering 43% of NY State retail jobs. It also keeps stale titles (`516` is still "Internet Publishing and Broadcasting", a NAICS 2002 title; NAICS 2022 reuses 516 for "Broadcasting and Content Providers") and has no `513`. 2-digit sectors are complete and safe; 3/4-digit detail is not, for retail and information from 2022.
- **Ownership:** at the all-industry line, Private + Government = Total and Federal + State + Local = Government, to within ±1 job (rounding). But at sector level the file has no "Total, All Ownerships" row for most sectors (e.g. Construction, Finance, Health care). Sector totals must be built by adding Private + Federal + State + Local. The Private rows are complete: sectors add up to the private total within ±3 jobs every year.

## A5. Coverage (NYC and boroughs)

In [8]:
b = q[(q.Area.isin(BORO_COUNTY)) & (q["Area Type"] == "County")].copy()
print("Borough rows:", len(b), "| years", b.Year.min(), "-", b.Year.max())
print("Rows per borough, 2024:", b[b.Year == "2024"].Area.value_counts().to_dict())
# Five boroughs add up to NYC?
e = lambda d: d[(d.NAICS == "00") & (d.Ownership == "Total, All Ownerships")].assign(emp=lambda x: pd.to_numeric(x["Average Employment"])).groupby("Year").emp.sum()
chk = pd.DataFrame({"boroughs_sum": e(b), "nyc_row": e(nyc)}).loc[YEARS]
chk["diff"] = chk.boroughs_sum - chk.nyc_row
display(chk)
# Sector-level: how many borough x sector x year cells exist vs a full grid (suppression is worse in small boroughs)
bs = b[(b.NAICS.str.len() == 2) & (b.NAICS != "00") & (b.Ownership == "Private") & b.Year.isin(YEARS)]
print("Private sector cells present per borough (max = 20 sectors x 7 years = 140):")
print(bs.groupby("Area").size().to_dict())

Borough rows: 73672 | years 2000 - 2025
Rows per borough, 2024: {'New York County': 607, 'Kings County': 602, 'Queens County': 565, 'Bronx County': 430, 'Richmond County': 427}


,boroughs_sum,nyc_row,diff
Year,,,
2019,4492711,4492708,3
2020,3997688,3997689,-1
2021,4065780,4065781,-1
2022,4347361,4347360,1
2023,4485695,4485695,0
2024,4630196,4630197,-1
2025,4652929,4652929,0


Private sector cells present per borough (max = 20 sectors x 7 years = 140):
{'Bronx County': 126, 'Kings County': 137, 'New York County': 133, 'Queens County': 131, 'Richmond County': 127}


**Coverage findings**
- Every county is in the file 2000–2025. The five boroughs add up to the NYC row within ±3 jobs every year .
- At the private sector × borough level, 6–14 of 140 cells (2019–25) are missing per borough: suppression bites in the Bronx and Staten Island (small sectors like mining, utilities, agriculture). Missing = suppressed, not zero.

## A6. Relationships (will it join?)

The industry → job bridge (BLS matrix, notebook 09) uses NAICS-based industry codes, so QCEW's NAICS codes need to be valid NAICS. Here: are QCEW's codes consistent across levels, and are the same codes used by CES and BEA (below)?

In [9]:
# Every 3-digit code sits under a 2-digit sector present the same year/area/ownership?
n3 = nyc[nyc.NAICS.str.len() == 3]
has_parent = n3.merge(nyc[nyc.NAICS.str.len() == 2][["Ownership", "Year", "NAICS"]].rename(columns={"NAICS": "sector"}),
                      on=["Ownership", "Year", "sector"], how="left", indicator=True)
print("NYC 3-digit rows whose 2-digit parent row is missing:", (has_parent._merge == "left_only").sum(), "of", len(n3))
# Sum of 3-digit children vs parent (only where children are complete is this exact)
p = nyc[nyc.Year.isin(YEARS) & (nyc.Ownership == "Private")].assign(emp=lambda x: pd.to_numeric(x["Average Employment"]))
kids = p[p.NAICS.str.len() == 3].groupby(["Year", "sector"]).emp.sum()
par = p[(p.NAICS.str.len() == 2)].set_index(["Year", "NAICS"]).emp
cmp_ = pd.concat([par.rename("sector_row"), kids.rename("sum_3digit")], axis=1).dropna()
cmp_["hidden_pct"] = (1 - cmp_.sum_3digit / cmp_.sector_row) * 100
print("Share of a sector's NYC private jobs hidden at 3-digit level (2024):")
display(cmp_.xs("2024").sort_values("hidden_pct", ascending=False).round(1).head(10))

NYC 3-digit rows whose 2-digit parent row is missing: 895 of 4392
Share of a sector's NYC private jobs hidden at 3-digit level (2024):


,sector_row,sum_3digit,hidden_pct
44,294934.0,101632.0,65.5
51,216299.0,150029.0,30.6
21,115.0,85.0,26.1
52,352445.0,349769.0,0.8
11,483.0,482.0,0.2
48,173358.0,173336.0,0.0
23,139379.0,139379.0,0.0
22,15975.0,15975.0,0.0
54,437038.0,437038.0,0.0
42,122357.0,122357.0,0.0


**Relationship findings**
- 895 of 4,392 NYC 3-digit rows have no 2-digit parent row in the same ownership (e.g. a Federal 3-digit row under a sector with no Federal sector row). Use Private only for drill-downs.
- Inside NYC private jobs, 3-digit detail is complete for most sectors (construction, professional services, wholesale: 0% hidden) but 65% of retail and 31% of information jobs have no 3-digit row in 2024, the NAICS 2022 gap above.
- Implication for notebook 09 (industry → job bridge): the BLS matrix works at NAICS 3–4 digit. If the bridge needs NYC industry detail below the sector, take it from BLS's own QCEW files (`data.bls.gov/cew`, with disclosure codes and NAICS 2022) rather than this extract. For v1's 2-digit "way in by industry", this file is enough.

## A7. Tie-out

(a) Against BLS. QCEW is a joint program, so BLS publishes the same county numbers. BLS QCEW open data for 2024, New York County, private, all industries ([BLS QCEW data files](https://www.bls.gov/cew/downloadable-data-files.htm)): the reference values below were read from BLS's county *annual averages* CSV (`https://data.bls.gov/cew/data/api/2024/a/area/36061.csv`, `own_code = 5`, `industry_code = 10`) on 2026-10-12.

In [10]:
# Reference values typed from the BLS QCEW open-data CSV (see markdown above); filled in pass 2
BLS_REF = {"Establishments": 128_155, "Average Employment": 2_240_692, "Total Wage": 383_844_839_182, "Annual Average Salary": 171_306}
ny24 = q[(q.Area == "New York County") & (q.Year == "2024") & (q.NAICS == "00") & (q.Ownership == "Private")]
display(ny24[["Area", "Ownership", "Year", "Establishments", "Average Employment", "Total Wage", "Annual Average Salary"]])
for k, v in BLS_REF.items():
    got = int(ny24[k].iloc[0])
    print(f"{k:<22} file {got:>16,} | BLS {v:>16,} | diff {got - v:+,}")
# (b) Internal: Total Wage / Average Employment = Annual Average Salary?
d = q[q.Year.isin(YEARS)].copy()
calc = pd.to_numeric(d["Total Wage"]) / pd.to_numeric(d["Average Employment"]).replace(0, pd.NA)
diff = (calc - pd.to_numeric(d["Annual Average Salary"])).abs()
print(f"\nSalary = wage / employment within $1: {(diff <= 1).mean():.2%} of 2019+ rows with employment > 0")

,Area,Ownership,Year,Establishments,Average Employment,Total Wage,Annual Average Salary
67332,New York County,Private,2024,128645,2240745,383920101199,171336


Establishments         file          128,645 | BLS          128,155 | diff +490
Average Employment     file        2,240,745 | BLS        2,240,692 | diff +53
Total Wage             file  383,920,101,199 | BLS  383,844,839,182 | diff +75,262,017
Annual Average Salary  file          171,336 | BLS          171,306 | diff +30



Salary = wage / employment within $1: 100.00% of 2019+ rows with employment > 0


**Tie-out findings**
- **Matches BLS** for Manhattan 2024 private, all industries: jobs 2,240,745 vs 2,240,692 (+53, 0.002%), wages +0.02%, average pay $171,336 vs $171,306. Establishments differ by +490 (0.4%). Most likely the two sites carry different revision dates; well inside what matters here.
- `Annual Average Salary` = `Total Wage / Average Employment` for 100% of 2019+ rows, so it can be recomputed and doesn't need to be stored.

---
# Part B · CES (Current Employment Statistics), NYC

## B0. Source header

| | NYS DOL CES, state and area (`nysdol_ces`) |
|---|---|
| **Publisher** | NYS Department of Labor with BLS (CES State and Metro Area program) |
| **Dataset** | *Current Employment Statistics: Beginning 1990* (data.ny.gov [`6k74-dgkb`](https://data.ny.gov/d/6k74-dgkb)) |
| **How it was collected** | Monthly survey of employer payrolls (jobs on the payroll in the pay period that includes the 12th). Benchmarked once a year to QCEW, so after the benchmark the history is revised to QCEW levels ([BLS SAE overview](https://www.bls.gov/sae/)) |
| **Covers** | Nonfarm wage and salary jobs, not seasonally adjusted here. Counts jobs, not people |
| **Codes** | CES industry codes: 8 digits = 2-digit *supersector* + NAICS-based detail. Not plain NAICS |
| **License** | NY Open Data |

In [11]:
CES = RAW / man.loc[man.key == "nysdol_ces", "file"].iloc[0]
c = pd.read_csv(CES, dtype=str, keep_default_na=False)
print(f"{len(c):,} rows x {c.shape[1]} cols | columns: {list(c.columns)}")
print("Non-digit employment cells:", (~c["Current Employment"].str.fullmatch(r"\d+")).sum(),
      c.loc[~c["Current Employment"].str.fullmatch(r"\d+"), "Current Employment"].value_counts().head().to_dict())
print("Areas:", c.Area.nunique(), "| NYC area code 93561:", c[c.Area == "93561"]["Area Name"].unique())
print("Key area + series + year + month duplicated:", c.duplicated(["Area", "Series", "Year", "Month"]).sum())
print("Month values:", sorted(c.Month.unique(), key=int))

498,400 rows x 7 cols | columns: ['Area', 'Area Name', 'Series', 'Title', 'Year', 'Month', 'Current Employment']


Non-digit employment cells: 0 {}
Areas: 43 | NYC area code 93561: <StringArray>
['New York City']
Length: 1, dtype: str
Key area + series + year + month duplicated: 0
Month values: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']


**Load findings:** 498,400 rows, 43 areas, no symbols, no duplicates on `Area + Series + Year + Month`. NYC is area `93561`.

## B2–B4. Grain, profile and codes (NYC)

Check: what is `Month = 0`? Is it the annual average of months 1–12?

In [12]:
cn = c[c.Area == "93561"].copy()
cn["emp"] = pd.to_numeric(cn["Current Employment"])
cn["y"] = cn.Year.astype(int); cn["m"] = cn.Month.astype(int)
print(f"NYC rows: {len(cn):,} | series: {cn.Series.nunique()} | years {cn.y.min()}-{cn.y.max()}")
print("Latest month:", cn.loc[cn.y == cn.y.max(), "m"].max())

ann = cn[cn.m == 0].set_index(["Series", "y"]).emp
mon = cn[cn.m > 0].groupby(["Series", "y"]).agg(mean=("emp", "mean"), n=("m", "nunique"))
j = mon.join(ann.rename("month0"), how="inner")
full = j[j.n == 12]
print(f"Month 0 = mean of 12 months (within 100 jobs): {((full["mean"] - full.month0).abs() <= 100).mean():.1%} of {len(full)} series-years")
print("Units: total nonfarm NYC 2024 month 0 =", f"{ann.get(('00000000', 2024), 'n/a'):,}")

# Series code structure: first 2 digits = supersector
cn["supersector"] = cn.Series.str[:2]
s = cn[["Series", "Title"]].drop_duplicates()
print("\nSeries with more than one title:", s.Series.duplicated().sum())
print("Supersector-level series (xx000000):")
display(s[s.Series.str[2:] == "000000"].sort_values("Series"))
print("Series per supersector:", cn.drop_duplicates("Series").supersector.value_counts().sort_index().to_dict())

NYC rows: 48,914 | series: 105 | years 1990-2026
Latest month: 8
Month 0 = mean of 12 months (within 100 jobs): 100.0% of 3698 series-years
Units: total nonfarm NYC 2024 month 0 = 4,793,700

Series with more than one title: 0
Supersector-level series (xx000000):


,Series,Title
18230,00000000,Total Nonfarm
18243,05000000,Total Private
18256,06000000,Goods Producing
18269,07000000,Service-Providing
18282,08000000,Private Service Providing
18295,15000000,"Mining, Logging and Construction"
8592,30000000,Manufacturing
8605,31000000,Durable Goods
8618,32000000,Non-Durable Goods
8644,40000000,"Trade, Transportation, and Utilities"


Series per supersector: {'00': 1, '05': 1, '06': 1, '07': 1, '08': 1, '15': 1, '20': 1, '30': 1, '31': 1, '32': 2, '40': 1, '41': 2, '42': 7, '43': 8, '50': 7, '55': 13, '60': 15, '65': 19, '70': 12, '80': 2, '90': 8}


## B5. Coverage: is every series continuous from 2019?

In [13]:
span = cn[cn.m > 0].groupby("Series").agg(first=("y", "min"), last=("y", "max"), n=("y", "size"))
span["expected"] = (span["last"] - span["first"]) * 12 + 12  # rough: assumes full first/last years
print("Series that start after 2019:", (span["first"] > 2019).sum())
print("Series that end before the latest year:", (span["last"] < cn.y.max()).sum())
display(span[(span["first"] > 2019) | (span["last"] < cn.y.max())].join(s.set_index("Series")).head(20))

Series that start after 2019: 0
Series that end before the latest year: 0


,first,last,n,expected,Title
Series,,,,,


**Grain, codes and coverage findings**
- One row = area × series × year × month. `Month = 0` is the annual average: it equals the mean of the 12 months for 100% of complete NYC series-years.
- Employment is in jobs, rounded to the hundred (NYC total nonfarm 2024: 4,793,700).
- NYC has 105 series, 1990 → August 2026. All 105 run unbroken from 2019 to now. The latest year is partial (Jan–Aug), and recent months are preliminary until the next benchmark.
- Series codes are CES industry codes: 2 digits of supersector + 6 digits of detail (`60541500` = Professional & business services → NAICS 5415 Computer systems design). Supersectors are not NAICS sectors: *Financial activities* = NAICS 52 + 53, *Private education & health* = 61 + 62, and there's no stand-alone NYC construction series (it's in `15000000` *Mining, logging and construction*).
- There is no NYC-by-borough CES.

## B6–B7. Relationships and tie-out: CES vs QCEW

CES is benchmarked to QCEW, so NYC annual-average CES total nonfarm should sit very close to QCEW total covered jobs (QCEW also includes a few farm jobs and the two differ in scope slightly).

In [14]:
qc_tot = nyc[(nyc.NAICS == "00") & (nyc.Ownership == "Total, All Ownerships")].set_index("Year")["Average Employment"].astype(int)
ces_tot = cn[(cn.Series == "00000000") & (cn.m == 0)].set_index("Year").emp
t = pd.DataFrame({"QCEW": qc_tot, "CES": ces_tot}).dropna().loc[lambda x: x.index.isin(YEARS)]
t["diff"] = t.CES - t.QCEW; t["diff_pct"] = (t["diff"] / t.QCEW * 100).round(2)
display(t)

# Sector level: CES supersector vs QCEW NAICS sector, 2024 (a few that map 1:1)
PAIRS = {"Mining, logging & construction": ("15000000", ["21", "23"]), "Information": ("50000000", ["51"]),
         "Financial activities": ("55000000", ["52", "53"]), "Prof. & business services": ("60000000", ["54", "55", "56"]),
         "Private educ. & health": ("65000000", ["61", "62"]), "Leisure & hospitality": ("70000000", ["71", "72"])}
rows = []
for name, (ser, naics) in PAIRS.items():
    qv = nyc[(nyc.Year == "2024") & (nyc.Ownership == "Private") & nyc.NAICS.isin(naics)]["Average Employment"].astype(int).sum()
    cv = cn[(cn.Series == ser) & (cn.y == 2024) & (cn.m == 0)].emp.sum()
    rows.append({"supersector": name, "CES": cv, "QCEW private": qv, "diff_pct": round((cv - qv) / qv * 100, 2)})
display(pd.DataFrame(rows))

,QCEW,CES,diff,diff_pct
Year,,,,
2019,4492708.0,4661400,168692.0,3.75
2020,3997689.0,4164800,167111.0,4.18
2021,4065781.0,4261200,195419.0,4.81
2022,4347360.0,4570100,222740.0,5.12
2023,4485695.0,4686700,201005.0,4.48
2024,4630197.0,4793700,163503.0,3.53
2025,4652929.0,4817500,164571.0,3.54


,supersector,CES,QCEW private,diff_pct
0,"Mining, logging & construction",142800,139494,2.37
1,Information,224000,216299,3.56
2,Financial activities,505900,483237,4.69
3,Prof. & business services,796900,751704,6.01
4,Private educ. & health,1258700,1181866,6.50
5,Leisure & hospitality,447900,442610,1.20


**Tie-out findings**
- **CES is 3.5–5.1% above QCEW** in every year (≈165,000–220,000 jobs). That's expected, not an error: CES counts nonfarm jobs that QCEW leaves out because they're not covered by unemployment insurance (students working for their school, religious organizations, some nonprofits, railroads), and it's benchmarked to a QCEW-*based* universe that adds them back ([BLS CES benchmark article](https://www.bls.gov/web/empsit/cesbmart.htm)). The gap is largest in education & health (+6.5%) and business services (+6.0%), smallest in leisure & hospitality (+1.2%).
- **Rule for the dashboard:** never put CES and QCEW numbers in one chart without saying which. Use QCEW for levels (it's a count, and it has boroughs and pay) and CES for the latest months (QCEW lags ~6 months).

---
# Part C · BEA GDP by county and industry

## C0. Source header

| | BEA CAGDP2 (current dollars) and CAGDP9 (real, chained 2017 dollars) (`bea_county_gdp`) |
|---|---|
| **Publisher** | U.S. Bureau of Economic Analysis, Regional Economic Accounts |
| **Dataset** | *GDP by county, metro, and other areas* ([BEA](https://www.bea.gov/data/gdp/gdp-county-metro-and-other-areas)) |
| **How it was collected** | **Model estimate.** Mostly allocates state GDP to counties using QCEW wages and other county data; not a count |
| **Codes** | BEA line codes over NAICS 2017 sectors (footnote 1), some lines are combinations (e.g. `52,53`) |
| **Suppression** | `(D)` = withheld to avoid disclosing a single firm; (NA) / (L) / (NM) per BEA conventions |
| **Revisions** | Re-estimated every year (this file: released Feb 5, 2026; 2024 new; 2020–2023 revised) |
| **License** | Public domain (U.S. government) |

In [15]:
BEA = {t: RAW / f for t, f in zip(["CAGDP2", "CAGDP9"], sorted(man.loc[man.key == "bea_county_gdp", "file"]))}
def read_bea(table):
    z = zipfile.ZipFile(BEA[table])
    name = next(n for n in z.namelist() if n.startswith(f"{table}_NY_"))
    d = pd.read_csv(z.open(name), dtype=str, keep_default_na=False, encoding="latin-1")
    print(f"{table}: member {name}, {len(d):,} rows x {d.shape[1]} cols")
    return d

g2, g9 = read_bea("CAGDP2"), read_bea("CAGDP9")
print("GeoFIPS raw values look like:", repr(g2.GeoFIPS.iloc[0]))
foot = g2[~g2.LineCode.str.strip().str.fullmatch(r"\d+")]
print("Rows without a line code (footer text):"); display(foot.iloc[:, :2])
for d in (g2, g9):
    d.drop(d[~d.LineCode.str.strip().str.fullmatch(r"\d+")].index, inplace=True)
    d["fips"] = d.GeoFIPS.str.replace('"', "").str.strip()
    for col in ["LineCode", "IndustryClassification", "Description", "Unit", "GeoName"]:
        d[col] = d[col].str.strip()
YCOLS = [str(y) for y in range(2001, 2025)]

CAGDP2: member CAGDP2_NY_2001_2024.csv, 2,146 rows x 32 cols
CAGDP9: member CAGDP9_NY_2001_2024.csv, 2,146 rows x 32 cols
GeoFIPS raw values look like: ' "36000"'
Rows without a line code (footer text):


,GeoFIPS,GeoName
2142,Note: See the included footnote file.,
2143,CAGDP2: Gross domestic product (GDP) by county 1.,
2144,"Last updated: February 5, 2026-- new statistics for 2024; revised ...",
2145,U.S. Bureau of Economic Analysis,


**Load findings:** each zip holds a CSV per state plus an all-areas file; we read the NY one (2,146 rows). `GeoFIPS` comes padded and quoted (`' "36061"'`), and the last four rows are footer text, both dropped here.

## C2–C4. Grain, profile, codes and special values

In [16]:
for name, d in [("CAGDP2", g2), ("CAGDP9", g9)]:
    print(f"{name}: duplicates on fips + LineCode = {d.duplicated(['fips', 'LineCode']).sum()} | areas {d.fips.nunique()} | lines {d.LineCode.nunique()} | units {d.Unit.unique().tolist()}")
bb = g2[g2.fips.isin(BOROUGHS)]
vals = bb[YCOLS].stack()
print("\nNon-numeric cells, NYC boroughs, CAGDP2:", vals[~vals.str.fullmatch(r"-?\d+")].value_counts().to_dict())
vals9 = g9[g9.fips.isin(BOROUGHS)][YCOLS].stack()
print("Non-numeric cells, NYC boroughs, CAGDP9:", vals9[~vals9.str.fullmatch(r"-?\d+")].value_counts().to_dict())
print("\nLine codes (NYC):")
display(bb[bb.fips == "36061"][["LineCode", "IndustryClassification", "Description"]])

CAGDP2: duplicates on fips + LineCode = 0 | areas 63 | lines 34 | units ['Thousands of dollars']
CAGDP9: duplicates on fips + LineCode = 0 | areas 63 | lines 34 | units ['Thousands of chained 2017 dollars']

Non-numeric cells, NYC boroughs, CAGDP2: {'(D)': 566}
Non-numeric cells, NYC boroughs, CAGDP9: {'(D)': 566}

Line codes (NYC):


,LineCode,IndustryClassification,Description
1054,1,...,All industry total
1055,2,...,Private industries
1056,3,11,"Agriculture, forestry, fishing and hunting"
1057,6,21,"Mining, quarrying, and oil and gas extraction"
1058,10,22,Utilities
1059,11,23,Construction
1060,12,31-33,Manufacturing
1061,13,"321,327-339",Durable goods manufacturing
1062,25,"311-316,322-326",Nondurable goods manufacturing
1063,34,42,Wholesale trade


**Grain and code findings**
- One row = county × industry line, with years across 24 columns (2001–2024). Key `fips + LineCode` unique in both tables.
- 34 line codes: the total, private, 20 sectors and 12 combined or sub-lines (`50` = 52 + 53, `13` = durable goods…). Summing all 34 lines double-counts. Only the 20 sector lines add up to the total.
- CAGDP2 = thousands of current dollars; CAGDP9 = thousands of chained 2017 dollars. Chained dollars don't add up across industries (BEA's own warning), so real GDP is for trends of one line, not for shares.
- The only special value in the NYC boroughs is `(D)` (withheld), 566 cells per table.

## C5. Coverage: which borough × industry cells are withheld (2019–2024)?

In [17]:
Y = [y for y in YCOLS if y >= "2019"]
sup = bb.set_index(["GeoName", "Description"])[Y].apply(lambda s: s.str.fullmatch(r"\(D\)"))
summary = sup.sum(axis=1).loc[lambda s: s > 0].unstack(0).fillna(0).astype(int)
print("Years withheld (of 6) by industry line x borough:"); display(summary)

Years withheld (of 6) by industry line x borough:


GeoName,"Bronx, NY","New York, NY","Richmond, NY"
Description,,,
"Agriculture, forestry, fishing and hunting",3,2,2
"Mining, quarrying, and oil and gas extraction",2,1,3
Wholesale trade,1,0,0
Transportation and warehousing,6,3,0
"Finance, insurance, real estate, rental, and leasing",3,0,2
Real estate and rental and leasing,3,0,2
Natural resources and mining,5,3,5
Trade,1,0,0
Transportation and utilities,6,3,1


**Coverage findings:** withholding hits the Bronx (transportation & warehousing withheld all 6 years) and Staten Island (manufacturing 4 of 6), plus tiny sectors everywhere. Brooklyn and Queens have no withheld cells 2019–24. Manhattan's withheld cells are all ≤ 2021.

## C6–C7. Relationships and tie-out

In [18]:
num = lambda d: d[Y].apply(pd.to_numeric, errors="coerce")
# (a) Boroughs sum to... NYC isn't a BEA county; compare to the NY metro? BEA has no NYC row. Check the state total instead:
st = g2[(g2.fips == "36000") & (g2.LineCode == "1")]
boro_tot = num(bb[bb.LineCode == "1"]).sum()
print("NYC (5 boroughs) share of NY State GDP:"); print((boro_tot / num(st).iloc[0] * 100).round(1).to_dict())
# (b) Industry lines sum to the total? (sectors only, excluding combined/summary lines)
SECT = ["3", "6", "10", "11", "12", "34", "35", "36", "45", "51", "56", "60", "64", "65", "69", "70", "76", "79", "82", "83"]
man36061 = g2[(g2.fips == "36061")]
tot = num(man36061[man36061.LineCode == "1"]).iloc[0]
sec = num(man36061[man36061.LineCode.isin(SECT)]).sum()
print("\nManhattan: total vs sum of sector lines (gap = withheld cells):")
display(pd.DataFrame({"total": tot, "sum_sectors": sec, "gap_pct": ((tot - sec) / tot * 100).round(2)}))
# (c) Real vs nominal: implied price index = current / real * 100 (2017 = 100)
nom = num(bb[bb.LineCode == "1"]).sum(); real = num(g9[g9.fips.isin(BOROUGHS) & (g9.LineCode == "1")]).sum()
print("NYC implied price index (2017 = 100):", (nom / real * 100).round(1).to_dict())
# (d) GDP per QCEW job by borough, 2024 (sanity: Manhattan should be far above the rest)
qj = b[(b.Year == "2024") & (b.NAICS == "00") & (b.Ownership == "Total, All Ownerships")].set_index("Area")["Average Employment"].astype(int)
gdp24 = bb[bb.LineCode == "1"].set_index("GeoName")["2024"].astype(int) * 1000
gdp24.index = [n.split(",")[0] + " County" for n in gdp24.index]
print("\nGDP per covered job, 2024:"); print((gdp24 / qj).round(-3).to_dict())

NYC (5 boroughs) share of NY State GDP:
{'2019': 59.6, '2020': 59.0, '2021': 59.2, '2022': 59.2, '2023': 59.0, '2024': 59.3}

Manhattan: total vs sum of sector lines (gap = withheld cells):


,total,sum_sectors,gap_pct
2019,779993754,7.765131e+08,0.45
2020,774908736,7.716152e+08,0.43
2021,851953791,8.484003e+08,0.42
2022,903778038,9.037780e+08,0.00
2023,935665528,9.356655e+08,-0.00
2024,1006673239,1.006673e+09,0.00


NYC implied price index (2017 = 100): {'2019': 105.0, '2020': 106.7, '2021': 110.0, '2022': 114.8, '2023': 119.9, '2024': 124.9}



GDP per covered job, 2024:
{'Bronx County': 175000.0, 'Kings County': 158000.0, 'New York County': 408000.0, 'Queens County': 186000.0, 'Richmond County': 174000.0}


**Relationship and tie-out findings**
- NYC (5 boroughs) = ~59% of New York State GDP, steady 2019–24.
- In Manhattan, the 20 sector lines add up to the total exactly in 2022–24; the 0.4% gap in 2019–21 is the withheld cells.
- The implied NYC price index (current ÷ real) rises from 105.0 (2019) to 124.9 (2024), so about +19% inflation over the period. That's in line with CPI (notebook 12), a useful cross-check later.
- **GDP per covered job:** Manhattan ≈ $408,000, the other four boroughs $158,000–$186,000. Expected: GDP is counted where the work is done, and Manhattan concentrates finance and headquarters.
- BEA's industry lines are NAICS 2017 sectors, so they join to QCEW's 2-digit sectors through a small hand-made line → sector table (20 lines, 1:1 except the combined lines, which we drop).

---
## 8. Fitness for purpose

| Question | Source | Fit | Why |
|---|---|---|---|
| Q1: which NYC industries are big and growing (by industry, 2019 → latest) | QCEW | Use | A count of every covered job, by sector, by borough, every year 2000–2025. Ties to BLS. 2-digit sectors complete |
| Q4: what an industry pays in NYC | QCEW | yes | Average annual pay by sector and borough. It's the average of all workers, not a starting wage, and it's pulled up by bonuses (Manhattan finance) |
| Industry detail below the sector (3–4 digit) | QCEW (this extract) | partial | Complete for most sectors, but retail and information lose most detail from 2022 (NAICS 2022 codes dropped). If needed, use BLS's QCEW files instead |
| **Q1: the latest months** | CES | for "now" | Monthly to Aug 2026, ~6 months ahead of QCEW. NYC only (no boroughs), supersectors, and preliminary until benchmarked |
| Industry jobs, as one consistent level | CES | partial | 3.5–5% above QCEW by definition. Don't mix with QCEW in one chart |
| Q1: which industries drive the NYC economy | BEA GDP | Context only | Output by borough × sector, 2001–2024. A model estimate; withheld cells in the Bronx and Staten Island; real GDP doesn't add up across industries. Useful as a "size of the industry" side panel, not as a job measure |
| Jobs by occupation inside an industry | — | no | None of the three knows occupations. That's the job of the BLS matrix (notebook 09) |
| Hiring, separations, new-hire pay, by age or education | — | no | QWI (notebook 08) |

**Recommendation for v1:** QCEW is the backbone of the "way in by industry": 2-digit sectors × borough × year, employment, establishments, wages. CES adds a "latest month" line for NYC. BEA is optional context and can wait until after v1 if time is short.

## 9. Issues list and proposed staging tables

| # | Issue | Source · rows affected | Proposed treatment (decided in staging) |
|---|---|---|---|
| 1 | Padded column name (`Annual Average Salary` + ~150 spaces) | QCEW · 1 column | Strip names |
| 2 | NYC listed three times (Labor Market Area, MSA, WIR) with identical numbers | QCEW · 40,964 rows | Keep `Area Type = 'Labor Market Area'` for NYC; keep `County` for the five boroughs |
| 3 | Suppressed cells are missing rows, not symbols | QCEW · 6–14 of 140 private sector × borough cells (2019–25) | Treat absent as `NULL` (unknown), never 0. In marts, show "not published" |
| 4 | No "Total, All Ownerships" row for most sectors | QCEW · sector level | Use `Private` for industry views (the public sector is shown as its own ownership); build all-ownership sector totals in core by adding the four ownerships, never by reading Total rows |
| 5 | NAICS 2022 codes missing below the sector (retail 449, 455–459; information 513) and stale titles (`516`) | QCEW · 2022–25, retail and information 3/4-digit | v1 uses 2-digit sectors only. Flag `naics_detail_complete = false` for 44 and 51 from 2022. If 3/4-digit is needed later, switch to BLS QCEW open data |
| 6 | NAICS range codes written as the first code (`31`, `44`, `48`) | QCEW | Map to `31-33`, `44-45`, `48-49` in the industry dimension, with BEA's and QWI's labels |
| 7 | `Annual Average Salary` is derived (wage ÷ employment) | QCEW · all | Don't store; compute in core |
| 8 | `Month = 0` is the annual average | CES · 1 row per series-year | Store months 1–12 only; compute annual averages in core (or keep month 0 as `is_annual_average`) |
| 9 | Latest months preliminary; history revised every March at benchmark | CES · current year | `vintage` column from the manifest; re-download yearly after the benchmark |
| 10 | CES industry codes are not NAICS | CES · all 105 series | Hand-made map series → NAICS sector(s) for the 20 supersector series; detail series map by their last 6 digits |
| 11 | CES ≠ QCEW level (+3.5–5%) | CES vs QCEW | Different measures in the fact tables; never summed together |
| 12 | Years wide (24 columns); quoted, padded `GeoFIPS`; 4 footer rows | BEA · all | Unpivot to one row per year; strip quotes; drop footer rows |
| 13 | `(D)` withheld | BEA · 566 NYC cells per table | `NULL` + `is_withheld = true` |
| 14 | Combined lines double-count | BEA · 12 of 34 lines | Keep the total, private and 20 sector lines; drop combinations |
| 15 | Chained dollars don't add across industries | BEA CAGDP9 | Use real GDP only for one line's trend; shares from CAGDP2 |
| 16 | Revised every year | BEA · 2020–2023 revised in this release | `release` column ("2026-02-05") |

### Proposed staging tables

**`stg_nysdol__qcew_annual`**
- **Grain / key:** `(area_code, ownership, naics_code, year)`
- **Columns:** `area_code` (county FIPS for boroughs, `NYC` for the city), `area_name`, `ownership`, `naics_code`, `naics_level` (2/3/4), `naics_title`, `year`, `establishments` int, `employment` int, `total_wages` bigint, `naics_detail_complete` bool, `source_file`
- **Filters:** NYC (Labor Market Area copy) + 5 borough counties, years 2019–2025

**`stg_nysdol__ces_monthly`**
- **Grain / key:** `(series_code, year, month)`, NYC only
- **Columns:** `series_code`, `series_title`, `supersector_code`, `year`, `month` (1–12), `employment` int, `vintage`
- **Filters:** `Area = '93561'`, `Month` 1–12, years 2019 → latest

**`stg_bea__county_gdp`**
- **Grain / key:** `(county_fips, line_code, year)`
- **Columns:** `county_fips`, `county_name`, `line_code`, `naics_label` (BEA's `IndustryClassification`), `line_title`, `year`, `gdp_current_thousands` bigint, `gdp_real_chained2017_thousands` bigint, `is_withheld` bool, `release`
- **Filters:** the five borough FIPS, 22 lines (total, private, 20 sectors), years 2019–2024. CAGDP2 and CAGDP9 joined on `(fips, line, year)` (same key, same rows)

In [19]:
# Row accounting for the proposed staging filters, recomputed rather than typed
qs = q[q.Year.isin(YEARS) & (((q.Area == "New York City") & (q["Area Type"] == "Labor Market Area"))
                            | (q.Area.isin(BORO_COUNTY) & (q["Area Type"] == "County")))]
assert not qs.duplicated(["Area", "Ownership", "NAICS", "Year"]).any()
print(f"QCEW: {len(q):,} rows in -> {len(qs):,} out (NYC + 5 boroughs, 2019-2025)")

cs = cn[(cn.m > 0) & (cn.y >= 2019)]
assert not cs.duplicated(["Series", "y", "m"]).any()
print(f"CES:  {len(c):,} rows in -> {len(cs):,} out (NYC, months 1-12, 2019 -> {cs.y.max()}-{cs[cs.y == cs.y.max()].m.max():02d})")

KEEP = ["1", "2"] + SECT
bs_ = g2[g2.fips.isin(BOROUGHS) & g2.LineCode.isin(KEEP)]
print(f"BEA:  {len(g2):,} NY rows per table -> {len(bs_):,} borough x line rows x {len(Y)} years = {len(bs_) * len(Y):,} long rows per table")

QCEW: 1,163,386 rows in -> 24,114 out (NYC + 5 boroughs, 2019-2025)
CES:  498,400 rows in -> 9,660 out (NYC, months 1-12, 2019 -> 2026-08)
BEA:  2,142 NY rows per table -> 110 borough x line rows x 6 years = 660 long rows per table
